# Study Helper Agent

This notebook builds a small LangChain study assistant that uses relevant `course_notes.txt` content when available, otherwise creates model-generated study material, then creates a cheatsheet and study plan in parallel before saving them together in study_plan.txt

code cell below imports all the needed packages: 
- dotenv for calling AI model API key
- langchain boilerplates 

In [ ]:
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import (
    RunnableBranch,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

## 1. Project Paths and Model Setup

This cell prepares the files and model used by the whole notebook.

- `PROJECT_DIR` points to the `lodge_project` folder relative to own file's location
- `NOTES_FILE` is the source material the agent studies from (course_notes.txt)
- `OUTPUT_FILE` is where the final cheatsheet and study plan are saved (study_plan.txt)
- `load_dotenv(...)` loads your OpenAI API key from `.env`.
- `model` is the chat model used by the subagents and the final orchestrator agent.

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "lodge_project" and (PROJECT_DIR / "lodge_project").exists():
    PROJECT_DIR = PROJECT_DIR / "lodge_project"

NOTES_FILE = PROJECT_DIR / "course_notes.txt"
OUTPUT_FILE = PROJECT_DIR / "study_plan.txt"

load_dotenv(PROJECT_DIR / ".env")
model = ChatOpenAI(model="gpt-4o-mini")

## 2. Fallback Notes Chain

The fallback notes chain creates study material only when the course notes do not contain a keyword match for the requested topic. It is an LCEL() chain, `fallback_notes_prompt | model | StrOutputParser()`, just like the later subagent chains.

Its output becomes the `notes` input for the cheatsheet and study-plan subagents. The final result clearly labels this material as model-generated, so it is never presented as content from `course_notes.txt`.

In [ ]:
# This chain supplies study material only when course_notes.txt has no match.
fallback_notes_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You create accurate, beginner-friendly study material for a student.",
        ),
        (
            "human",
            """The local course notes do not cover this topic: {topic}

Create concise source material for the next study workflow. Explain key ideas,
plain definitions, important rules or formulas, and a simple example if useful.
Do not claim this material came from the course notes.""",
        ),
    ]
)

In [ ]:
fallback_notes_chain = fallback_notes_prompt | model | StrOutputParser()

## 3. Notes Finder Helpers

These helper functions are plain Python. They are not LangChain tools yet.

`_words(text)` turns text into lowercase words. This lets the search compare the user's topic with each line of notes using simple keyword matching.

`_read_notes()` opens `course_notes.txt`, removes blank lines, and returns the notes as a list of strings. Each line is treated as one small note.

`_matching_note_lines(query, notes)` scores each note by keyword overlap and returns only true matches. `_format_notes(notes)` turns those lines into prompt-ready bullet points. Keeping these as helper functions makes both `search_notes()` and the conditional chain shorter and easier to understand.

In [ ]:
def _words(text: str) -> set[str]:
    """Normalize text into lowercase words for simple keyword search."""
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def _read_notes() -> list[str]:
    """Read non-empty note lines from course_notes.txt."""
    if not NOTES_FILE.exists():
        return []
    return [line.strip() for line in NOTES_FILE.read_text().splitlines() if line.strip()]


def _matching_note_lines(query: str, notes: list[str]) -> list[str]:
    """Return the strongest note lines with at least one keyword match."""
    query_words = _words(query)
    scored = [
        (len(query_words & _words(note)), note)
        for note in notes
        if query_words & _words(note)
    ]
    return [note for _, note in sorted(scored, reverse=True)[:8]]


def _format_notes(notes: list[str]) -> str:
    """Format note lines consistently for tools and prompt context."""
    return "\n".join(f"- {note}" for note in notes)

## 4. Notes Finder Tool

`search_notes()` is the first real tool.

A tool is a function the agent can decide to call. The `@tool` decorator tells LangChain that this function is available to the model.

Purpose of this tool:

- Take the topic or subtopic the user wants to study.
- Search `course_notes.txt` for relevant note lines.
- Return the strongest matching notes as plain text.

This is intentionally a simple keyword search, not a vector database. For this demo, the goal is to teach tool use and agent orchestration without adding RAG complexity.

In [ ]:
@tool
def search_notes(query: str) -> str:
    """
    Find note lines that match the requested study topic.

    Use this first whenever the user asks to study a topic or revise a plan for
    a subtopic.
    """
    query_words = _words(query)
    notes = _read_notes()

    if not notes:
        return "No notes found. Add study notes to course_notes.txt."
    if not query_words:
        return "\n".join(notes[:8])

    matching_notes = _matching_note_lines(query, notes)
    if not matching_notes:
        return "No exact note matches found. Closest available notes:\n" + "\n".join(
            f"- {note}" for note in notes[:8]
        )

    return _format_notes(matching_notes)

## 5. Conditional Context Chain

This is the conditional chaining step. `context_with_matches` calculates the local keyword matches once. `RunnableBranch` then selects one of two context paths:

- When matches exist, it attaches formatted `course_notes.txt` lines and labels the source.
- When no matches exist, it invokes `fallback_notes_chain`, attaches its model-generated material, and labels that source.

Both paths produce the same input shape: the original request fields plus `notes` and `source`. This lets the downstream chains work without needing to know which path ran.


#### Original shape of input:

{
    
    "topic": "attention",

    "sessions": 1
}

#### New shape of input after context_with_matches

{
    
    "topic": "attention",

    "sessions": 1,

    "matching_notes": [
        "Attention helps a transformer decide which tokens matter most.",
        "Self-attention compares each token with other tokens."
    ]
}

In [ ]:
context_with_matches = RunnablePassthrough.assign(
    matching_notes=lambda inputs: _matching_note_lines(
        inputs["topic"], _read_notes()
    )
)

In [ ]:
# checks if matching_notes array is empty, then defines notes and source accordingly

conditional_context_chain = context_with_matches | RunnableBranch(
    (
        lambda inputs: bool(inputs["matching_notes"]),
        RunnablePassthrough.assign(
            notes=lambda inputs: _format_notes(inputs["matching_notes"]),
            source=lambda _: "course_notes.txt",
        ),
    ),
    RunnablePassthrough.assign(
        notes=fallback_notes_chain,
        source=lambda _: (
            "Model-generated study material because course_notes.txt had "
            "no relevant match."
        ),
    ),
)

## 6. Cheatsheet Chain

The Cheatsheet Chain is a small LCEL chain focused on one job: turn relevant notes into one beginner-friendly cheatsheet.

It has three parts:

- `cheatsheet_prompt`: a prompt template with slots for `{topic}` and `{notes}`.
- `cheatsheet_chain`: the runnable chain `prompt | model | StrOutputParser()`.
- `create_cheatsheet()`: a tool wrapper that lets the final orchestrator call this subagent.

The prompt tells the model to stay grounded in the selected `notes` context. That context comes from `course_notes.txt` when relevant, or from the labeled fallback notes chain when no relevant course material exists. This chain receives the same selected context as the Study Plan Subagent so both can run in parallel.

In [ ]:
cheatsheet_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You create concise student cheatsheets grounded only in provided notes.",
        ),
        (
            "human",
            """Topic: {topic}

Notes:
{notes}

Create one cheatsheet with:
- key ideas
- definitions
- formulas or rules if useful
- memory cues

If the notes do not contain enough information, say what is missing.""",
        ),
    ]
)

In [ ]:
cheatsheet_chain = cheatsheet_prompt | model | StrOutputParser()

In [ ]:
@tool
def create_cheatsheet(topic: str, notes: str) -> str:
    """
    Run the Cheatsheet Subagent chain.

    Use this after search_notes so the cheatsheet is grounded in course_notes.txt.
    """
    return cheatsheet_chain.invoke({"topic": topic, "notes": notes})

## 7. Study Plan Chain

The Study Plan Chain turns the relevant notes into an action plan. It does not depend on the cheatsheet, because that would force it to wait for the cheatsheet chain to finish.

It also has three parts:

- `study_plan_prompt`: instructions for splitting a topic into subtopics and tasks.
- `study_plan_chain`: another LCEL chain using `prompt | model | StrOutputParser()`.
- `create_study_plan()`: a tool wrapper so the orchestrator can call the chain.

The important teaching idea is the difference between `Content` and `Practice`:

- `Content` means facts, definitions, and explanations the student should understand or memorize.
- `Practice` means things the student must be able to do, solve, explain, trace, compare, or apply.

The default is one study session. If the user asks for more sessions, the final agent passes that session count into this tool. If the user asks for more practice on a subtopic, the final agent passes that subtopic as `practice_focus`. Both this chain and the cheatsheet chain receive the same topic and selected context.

In [ ]:
study_plan_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You create practical study plans with content tasks and practice tasks.",
        ),
        (
            "human",
            """Topic: {topic}
Number of study sessions: {sessions}
Subtopic needing more practice: {practice_focus}

Notes:
{notes}

Break the topic into subtopics, then create a study plan.
For each session, include:
- Content: information and notes to understand or commit to memory
- Practice: skills and common problem types the student must solve

Default to one session when the user does not specify a session count. If a
practice focus is provided, add more practice tasks for that subtopic.""",
        ),
    ]
)

In [ ]:
study_plan_chain = study_plan_prompt | model | StrOutputParser()

In [ ]:
@tool
def create_study_plan(
    topic: str,
    notes: str,
    sessions: int = 1,
    practice_focus: str = "none",
) -> str:
    """
    Run the Study Plan Subagent chain by itself.

    This teaching tool accepts searched notes directly. The normal workflow
    uses create_and_save_study_output to run this chain alongside the
    cheatsheet chain.
    """
    return study_plan_chain.invoke(
        {
            "topic": topic,
            "notes": notes,
            "sessions": sessions or 1,
            "practice_focus": practice_focus or "none",
        }
    )

## 8. Parallel Study Output Chain

`RunnableParallel` is the LCEL component that runs independent branches at the same time. It sends the context selected by the conditional chain to both subagents.

- The `cheatsheet` branch uses `topic` and `notes`.
- The `study_plan` branch uses `topic`, `notes`, `sessions`, and `practice_focus`.

Both chains accept the full dictionary and use only the prompt variables they need. When they finish, the parallel chain returns a dictionary with `cheatsheet` and `study_plan` keys.

In [ ]:
# RunnableParallel sends one shared input dictionary to both independent
# subagents. It returns {"cheatsheet": ..., "study_plan": ...} when both finish.
parallel_study_output_chain = RunnableParallel(
    cheatsheet=cheatsheet_chain,
    study_plan=study_plan_chain,
)

## 9. Save Output Tool

`save_study_output()` is a tool, but it is not a subagent because it does not call the model.

Purpose of this tool:

- Take the topic, source label, cheatsheet, and study plan.
- Write them into `study_plan.txt`.
- Return a short confirmation message.

This gives the demo a visible result outside the notebook. After the agent runs, the student can open `study_plan.txt` and keep the generated study material.

In [ ]:
@tool
def save_study_output(
    topic: str,
    cheatsheet: str,
    study_plan: str,
    source: str = "course_notes.txt",
) -> str:
    """
    Save the latest cheatsheet and study plan, including their source label.

    Use this every time a new or revised study plan is created.
    """
    OUTPUT_FILE.write_text(
        f"# Source\n\n{source}\n\n# {topic} Cheatsheet\n\n{cheatsheet}\n\n"
        f"# Study Plan\n\n{study_plan}\n"
    )
    return f"Saved cheatsheet and study plan to {OUTPUT_FILE.name}. Source: {source}"

## 10. Combined Study Output Tool

`create_and_save_study_output()` is the normal workflow tool. It keeps the final agent simple because one tool call handles the complete workflow.

It first runs `conditional_context_chain` to select matching course notes or model-generated fallback material. It then runs both subagent chains through `parallel_study_output_chain`, passes their paired outputs and source label to `save_study_output()`, and returns the complete result. The individual tools remain available for learning or for a user who explicitly asks to see one workflow step.

In [ ]:
@tool
def create_and_save_study_output(
    topic: str,
    sessions: int = 1,
    practice_focus: str = "none",
) -> str:
    """
    Create and save a complete cheatsheet and study plan for a topic.

    This is the normal workflow tool. It uses matching course notes when
    available, otherwise creates model-generated study material, then sends the
    selected context to both subagents in parallel and saves their paired output.
    """
    context = conditional_context_chain.invoke(
        {
            "topic": topic,
            "sessions": sessions or 1,
            "practice_focus": practice_focus or "none",
        }
    )
    output = parallel_study_output_chain.invoke(context)
    saved_message = save_study_output.invoke(
        {
            "topic": topic,
            "cheatsheet": output["cheatsheet"],
            "study_plan": output["study_plan"],
            "source": context["source"],
        }
    )
    return (
        f"{saved_message}\n\n# Source\n\n{context['source']}\n\n"
        f"# {topic} Cheatsheet\n\n{output['cheatsheet']}\n\n"
        f"# Study Plan\n\n{output['study_plan']}"
    )

## 11. Final Orchestrator Agent

The final orchestrator agent stays near the bottom because it depends on everything defined above.

This is the main agent the user talks to. It does not write the cheatsheet or plan directly by itself. Instead, it decides which tools to call. For normal requests it uses the combined workflow tool.

Available tools:

- `create_and_save_study_output`: normal path that selects course notes or fallback material, generates both outputs in parallel, and saves them with a source label.
- `search_notes`, `create_cheatsheet`, `create_study_plan`, and `save_study_output`: individual teaching tools for demonstrating the workflow steps.

`MessagesPlaceholder("chat_history")` gives the agent memory during the current notebook run. This is what lets a follow-up like "make it 3 sessions" revise the previous topic instead of starting over.

`MessagesPlaceholder("agent_scratchpad")` is where LangChain stores the agent's intermediate tool-call reasoning. The user does not manually fill it in; the agent executor manages it.

In [ ]:
def build_agent() -> AgentExecutor:
    """Build the main Study Planning Agent that coordinates the subagents."""
    tools = [
        search_notes,
        create_cheatsheet,
        create_study_plan,
        save_study_output,
        create_and_save_study_output,
    ]

    # MessagesPlaceholder("chat_history") gives the agent short-term memory.
    agent_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are a Study Planning Agent.

Your job:
1. Identify the study topic from the user.
2. Use create_and_save_study_output for every new or revised study request.
   It uses relevant course notes when available, otherwise creates model
   fallback material, runs the cheatsheet and study-plan subagents in parallel,
   and saves the result with a source label.
3. The individual tools are available only to explain or demonstrate each
   workflow step when the user explicitly asks about them.

Defaults:
- If the user does not specify study sessions, use 1 session.
- If the user asks for more sessions, revise the previous plan using chat history.
- If the user asks for more practice on a subtopic, revise the practice tasks for that subtopic.

Final answer format:
- Briefly state what was created or revised.
- Show the cheatsheet.
- Show the study plan.
- Mention that study_plan.txt was saved.""",
            ),
            MessagesPlaceholder("chat_history"),
            ("human", "{input}"),
            MessagesPlaceholder("agent_scratchpad"),
        ]
    )

    # create_tool_calling_agent lets the model choose which tools to call.
    agent = create_tool_calling_agent(model, tools, agent_prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=True)

## 12. CLI Runner

`main()` starts a simple terminal-style conversation inside the notebook.

What happens each turn:

1. The user types a request.
2. The request and `chat_history` are sent to the agent executor.
3. For a normal request, the agent calls the combined tool, which selects course notes or fallback material, runs both subagents in parallel, and saves output with its source.
4. The final answer is printed.
5. The user message and assistant answer are added to `chat_history`.

The memory is temporary. It lasts while this notebook session is running, but it is not saved to a database.

Example first prompt:

```text
I want to study transformers and attention.
```

Example follow-up prompts:

```text
Make it 3 study sessions.
Give me more practice on tokens, embeddings, and self-attention.
```

In [ ]:
def main() -> None:
    """Run a terminal chat loop with in-memory conversation history."""
    agent_executor = build_agent()
    chat_history = []

    print("Study Planning Agent")
    print("Ask for a topic cheatsheet and study plan. Type 'exit' to quit.\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() == "exit":
            break

        # Send all previous messages so the agent can revise prior plans.
        result = agent_executor.invoke(
            {"input": user_input, "chat_history": chat_history}
        )
        answer = result["output"]

        # Store this turn after the model answers, matching the chat model demo.
        chat_history.append(HumanMessage(content=user_input))
        chat_history.append(AIMessage(content=answer))

        print(f"\nAgent: {answer}\n")


if __name__ == "__main__":
    main()